In [1]:
# imports
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
DIR = Path("dataset")
files = sorted(DIR.rglob("*.parquet"))
print(len(files), "files")

2228 files


In [3]:
def load_instance(path: Path) -> pd.DataFrame:
    """Read each instance of Parquet file to add metadata"""
    df = pd.read_parquet(path)

    if isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()

    # Adding metadata
    file_name = path.stem.upper()
    df["instance_id"] = path.stem
    df["event_class"] = int(path.parent.name)

    # WELL = real well, SIMULATED = simulation, DRAWN = handdrawn
    df["source"] = ("simulated" if file_name.startswith("SIMULATED")
                    else "drawn" if file_name.startswith("DRAWN")
                    else "real")

    df["well_id"] = path.stem.split("_")[0] if file_name.startswith("WELL") else None

    return df


def reduce_memory(df: pd.DataFrame, save_path: str) -> pd.DataFrame:

    print(f"Total memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # float32
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].where(df[c].abs() <= np.finfo("float32").max).astype("float32")

    # metadata text to category
    for c in ["instance_id", "well_id", "source"]:
        df[c] = df[c].astype("category")
    df["event_class"] = df["event_class"].astype("int8")

    print(f"Reduced memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    df.to_parquet(save_path)
    return df


CACHE = Path("final.parquet")

if CACHE.exists():
    df = pd.read_parquet(CACHE)
    print("df loaded from cache.")
else:
    frames = []
    for i, f in enumerate(files):
        frames.append(load_instance(f))
        if i % 500 == 0:                     
            print(f"{i}/{len(files)}")

    df = pd.concat(frames, ignore_index=True)
    df = reduce_memory(df, save_path="final.parquet")
    del frames
    print("Loading completed.")


print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}\n\n")
df.head()

df loaded from cache.
Shape: (76587318, 34)

Column types:
timestamp        datetime64[ns]
ABER-CKGL               float32
ABER-CKP                float32
ESTADO-DHSV             float32
ESTADO-M1               float32
ESTADO-M2               float32
ESTADO-PXO              float32
ESTADO-SDV-GL           float32
ESTADO-SDV-P            float32
ESTADO-W1               float32
ESTADO-W2               float32
ESTADO-XO               float32
P-ANULAR                float32
P-JUS-BS                float32
P-JUS-CKGL              float32
P-JUS-CKP               float32
P-MON-CKGL              float32
P-MON-CKP               float32
P-MON-SDV-P             float32
P-PDG                   float32
PT-P                    float32
P-TPT                   float32
QBS                     float32
QGL                     float32
T-JUS-CKP               float32
T-MON-CKP               float32
T-PDG                   float32
T-TPT                   float32
class                     Int16
state        

,timestamp,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,...,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state,instance_id,event_class,source,well_id
0,2017-02-01 01:02:07,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,84.644630,NaN,0.0,119.078102,<NA>,<NA>,WELL-00001_20170201010207,0,real,WELL-00001
1,2017-02-01 01:02:08,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,84.638283,NaN,0.0,119.078102,<NA>,<NA>,WELL-00001_20170201010207,0,real,WELL-00001
2,2017-02-01 01:02:09,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,84.631943,NaN,0.0,119.078102,<NA>,<NA>,WELL-00001_20170201010207,0,real,WELL-00001
3,2017-02-01 01:02:10,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,84.625580,NaN,0.0,119.078102,<NA>,<NA>,WELL-00001_20170201010207,0,real,WELL-00001
4,2017-02-01 01:02:11,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,84.619232,NaN,0.0,119.078102,<NA>,<NA>,WELL-00001_20170201010207,0,real,WELL-00001
